# 🚀 MoodMax Emotion Model Training & Optimization Pipeline

This notebook runs the complete model improvement pipeline on Google Colab (Tesla T4 GPU).

### 🛡️ Non-Degradation Guarantee
- Your baseline model is **never overwritten** directly.
- Candidates are trained to `models/emotion-distilbert-multi-v2`.
- An automated **Anti-Degradation Gate** validates that Macro-F1 does not drop and high-performing anchor classes (`joy`, `neutral`) stay within acceptable variance before promotion.

## 1. Check GPU & Setup Environment

In [ ]:
!nvidia-smi

## 2. Clone Repository

In [ ]:
# Clone MoodMax and enter workspace
!git clone https://github.com/Patel-Priyank-1602/MoodMax.git
%cd MoodMax

## 3. Install Training Dependencies

In [ ]:
!pip install -q -r training/requirements-training.txt

## 4. Prepare Dataset (GoEmotions 7-class Ekman)

In [ ]:
# Download GoEmotions and collapse 28 labels into 7 Ekman emotion classes
!python data/scripts/download_goemotions.py
!python data/scripts/collapse_labels.py

## 5. Phase 1 — Per-Class Threshold Optimization (Fast Inference on T4)
Finds optimal per-class decision thresholds (recall-weighted F-beta=1.5).

In [ ]:
!python training/optimize_thresholds.py --eval_test

## 6. Phase 4 — Targeted Retraining with Rare-Class Oversampling
Trains `distilbert-base-multilingual-cased` with 2.5x oversampling for weak classes (`fear`, `disgust`, `sadness`) and class-weighted BCE loss.

In [ ]:
!python training/train_v2.py --batch_size 32 --epochs 8 --oversample_factor 2.5

## 7. Anti-Degradation Safety Gate & Evaluation
Compares candidate model (`models/emotion-distilbert-multi-v2`) against baseline.
Only promotes if Macro-F1 is higher and anchor classes did not degrade.

In [ ]:
!python training/compare_and_gate.py --candidate models/emotion-distilbert-multi-v2 --promote

## 8. Save Trained Model to Google Drive (Recommended)
Mounts your Google Drive and saves the 541MB model archive directly into a `MoodMax_Models` folder on your Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create target folder on Google Drive
!mkdir -p "/content/drive/MyDrive/MoodMax_Models"

# Zip and save directly to Google Drive
!zip -r "/content/drive/MyDrive/MoodMax_Models/emotion_model_best.zip" models/emotion-distilbert-multi/

print("\n✓ Model successfully saved to Google Drive: MyDrive/MoodMax_Models/emotion_model_best.zip")

## 9. (Optional) Direct Browser Download
If you want to trigger a direct download in your browser from the Drive archive:

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/MoodMax_Models/emotion_model_best.zip')